# 02 — Modeling: quantile XGBoost vs. baselines with temporal validation

This notebook trains the demand model **from scratch** on the committed synthetic
panel and evaluates it on a strictly temporal holdout. It does not read any
pre-computed artifact, so it is executable on a clean checkout of milestone M3.

1. Temporal validation design
2. Feature contract
3. Baselines: seasonal naive and Croston/SBA
4. XGBoost quantile models (`p10` / `p50` / `p90`)
5. Holdout results
6. Error analysis by ABC class
7. Feature importance
8. Prophet baseline (optional, sample series)
9. Decision, limitations and next steps


In [1]:
from __future__ import annotations

import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message=".*joblib will operate in serial mode.*")

from xgboost import XGBRegressor

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "src" / "data_contracts").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or from notebooks/.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_contracts import load_csv
from src.pipelines.features import FEATURES, SERIES_KEYS, build_supervised

DATA_PATH = REPO_ROOT / "data" / "processed" / "demand_daily.csv"
daily = load_csv(DATA_PATH, "demand_daily")
print(f"Panel: {DATA_PATH.relative_to(REPO_ROOT)}")
print(
    f"{len(daily):,} rows | {daily['date'].min()} -> {daily['date'].max()} | "
    f"{daily[list(SERIES_KEYS)].drop_duplicates().shape[0]} series"
)

Panel: data/processed/demand_daily.csv
65,760 rows | 2023-01-01 -> 2025-12-31 | 60 series


## 1. Temporal validation design

The panel has one row per `product × branch × day`. A random split would leak
the future into the past and inflate the score, so the evaluation is a
**rolling-origin style holdout**:

- The last **28 calendar days per series** (2025-12-04 → 2025-12-31) are the
  holdout; the model never sees them during fitting.
- The first 28 days of every series are used only as lag history and are
  excluded from training.
- Every feature is computed from data **strictly before** the target day
  (`lag_*`, `roll_*`, lagged promotions / stock-out flags / prices).
- Metrics are computed on the units scale after the inverse `log1p`.
- This mirrors production: train on the past, forecast the next four weeks.


In [2]:
train_df, valid_df, meta = build_supervised(daily, holdout_days=28)
valid_dates = pd.to_datetime(valid_df["date"])
print(f"Train rows: {len(train_df):,} | holdout rows: {len(valid_df):,}")
print(f"Holdout window: {valid_dates.min().date()} -> {valid_dates.max().date()}")
print(f"Series: {valid_df[list(SERIES_KEYS)].drop_duplicates().shape[0]} | features: {len(FEATURES)}")
print(f"Target transform: {meta['target_transform']} | min history: {meta['min_history_days']} days")

Train rows: 62,400 | holdout rows: 1,680
Holdout window: 2025-12-04 -> 2025-12-31
Series: 60 | features: 27
Target transform: log1p | min history: 28 days


## 2. Feature contract

The 27 features are the **serving contract**: the API and the training pipeline
must build exactly the same columns, in the same order, from the same history.

| Group | Features | Why it matters |
|---|---|---|
| Demand history | `lag_1/7/14/28`, `roll_mean_7/14/28`, `roll_std_7`, `days_since_last_sale` | Captures momentum, weekly repetition and intermittency. |
| Calendar | `dow`, `month`, `is_weekend`, `sin_doy`, `cos_doy` | Weekly and annual seasonality; known in advance. |
| Inventory & price (lagged) | `promo_lag1`, `stockout_flag_lag1`, `pending_units`, `unit_price_mxn`, `unit_cost_mxn`, `margin_pct`, `lead_time_days`, `stock_to_demand` | Commercial drivers and censoring; lagged to avoid leaking the target. |
| Categorical encodings | `product_code`, `branch_code`, `category_code`, `unit_code`, `demand_pattern_code` | Lets one global model share strength across the 60 series. |


In [3]:
FEATURE_GROUPS = {
    "Demand history": [
        "lag_1", "lag_7", "lag_14", "lag_28",
        "roll_mean_7", "roll_mean_14", "roll_mean_28", "roll_std_7",
        "days_since_last_sale",
    ],
    "Calendar": ["dow", "month", "is_weekend", "sin_doy", "cos_doy"],
    "Inventory & price (lagged)": [
        "promo_lag1", "stockout_flag_lag1", "pending_units", "unit_price_mxn",
        "unit_cost_mxn", "margin_pct", "lead_time_days", "stock_to_demand",
    ],
    "Categorical encodings": [
        "product_code", "branch_code", "category_code", "unit_code", "demand_pattern_code",
    ],
}
assert set(sum(FEATURE_GROUPS.values(), [])) == set(FEATURES)
for group, features in FEATURE_GROUPS.items():
    print(f"{group:<28} {len(features):>2} features")

preview_columns = [
    "date", "lag_1", "lag_7", "roll_mean_7", "pending_units", "stock_to_demand",
    "product_code", "y",
]
print("\nTraining-frame preview:")
print(train_df[preview_columns].head(3).round(3).to_string(index=False))

Demand history                9 features
Calendar                      5 features
Inventory & price (lagged)    8 features
Categorical encodings         5 features

Training-frame preview:
      date  lag_1  lag_7  roll_mean_7  pending_units  stock_to_demand  product_code     y
2023-01-29   2.07   0.00        1.139            0.0            6.098           0.0 0.000
2023-01-30   0.00   1.61        1.139            0.0            6.098           0.0 0.000
2023-01-31   0.00   0.10        0.909            0.0            6.832           0.0 1.004


## 3. Baselines

Two **deterministic, pre-holdout-only** baselines set the bar:

- **Seasonal naive (`lag_7`)**: predict the same weekday of the previous week.
  It is the default in most SMEs and has no training cost.
- **Croston/SBA** (`alpha = 0.1`): forecast a constant demand rate from the
  exponentially smoothed demand size and inter-demand interval. It is the
  classic baseline for intermittent SKUs.

A model that cannot beat these is not worth deploying.


In [4]:
def croston_sba(values, alpha: float = 0.1) -> float:
    """Croston with the Syntetos-Boylan approximation, fitted on history only."""
    size = None
    interval = None
    since_last = 0
    forecast = 0.0
    for raw in values:
        value = max(0.0, float(raw))
        since_last += 1
        if value <= 0:
            continue
        if size is None or interval is None:
            size = value
            interval = float(since_last)
        else:
            size = size + alpha * (value - size)
            interval = interval + alpha * (since_last - interval)
        since_last = 0
        forecast = max(0.0, (1.0 - alpha / 2.0) * size / max(interval, 1e-9))
    return forecast


def regression_metrics(actual, prediction, naive=None, p10=None, p90=None) -> dict:
    errors = np.abs(actual - prediction)
    mae = float(errors.mean())
    out = {
        "mae": round(mae, 4),
        "wape": round(float(errors.sum() / actual.sum()), 4),
    }
    if naive is not None:
        naive_mae = float(np.abs(actual - naive).mean())
        out["mase"] = round(mae / naive_mae, 4) if naive_mae else 0.0
    if p10 is not None and p90 is not None:
        out["coverage_90"] = round(float(((actual >= p10) & (actual <= p90)).mean()), 4)
    return out


valid = valid_df.copy()
valid["date"] = pd.to_datetime(valid["date"])
actual = valid["units_sold"].to_numpy(dtype=float)
naive = valid["lag_7"].to_numpy(dtype=float)

first_dates = valid.groupby(list(SERIES_KEYS), sort=False)["date"].min()
history = daily.copy()
history["date"] = pd.to_datetime(history["date"])
croston_by_series = {}
for key, group in history.groupby(list(SERIES_KEYS), sort=False):
    cutoff = first_dates.get(tuple(key))
    past = group[group["date"] < cutoff].sort_values("date") if cutoff is not None else group
    croston_by_series[tuple(key)] = croston_sba(past["units_sold"].to_numpy(dtype=float))
keys = list(zip(valid["business_id"], valid["branch_id"], valid["product_id"], strict=False))
croston = np.asarray([croston_by_series.get(key, 0.0) for key in keys], dtype=float)

print("Seasonal naive (lag_7):", regression_metrics(actual, naive))
print("Croston/SBA (alpha=0.1):", regression_metrics(actual, croston))

Seasonal naive (lag_7): {'mae': 2.4713, 'wape': 0.6785}
Croston/SBA (alpha=0.1): {'mae': 2.2163, 'wape': 0.6085}


## 4. XGBoost quantile models

Why a **global** model instead of one model per series:

- 60 series, many with intermittent or lumpy demand; a per-series Prophet/ARIMA
  overfits or cannot be estimated at all.
- One model shares strength across products and branches and handles cold-start
  series (e.g., a new branch) from the first forecast.
- The three quantile objectives (`reg:quantileerror` at 0.1 / 0.5 / 0.9) produce
  the interval the inventory policy needs, not just a point.

Training protocol: `log1p` target, histogram tree method, 300 trees, depth 6,
learning rate 0.05, deterministic seed. Predictions are made monotonic
(`p10 <= p50 <= p90`) before scoring.


In [5]:
QUANTILES = {"p10": 0.1, "p50": 0.5, "p90": 0.9}
X_train = train_df[FEATURES].to_numpy(dtype=float)
y_train = train_df["y"].to_numpy(dtype=float)
X_valid = valid_df[FEATURES].to_numpy(dtype=float)

models = {}
started = time.time()
for name, alpha in QUANTILES.items():
    model = XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=alpha,
        tree_method="hist",
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        min_child_weight=1.0,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=2,
        verbosity=0,
    )
    model.fit(X_train, y_train)
    models[name] = model
print(f"Trained {len(models)} quantile models in {time.time() - started:.1f}s")

raw = {name: model.predict(X_valid) for name, model in models.items()}
lower = np.minimum(raw["p10"], raw["p90"])
upper = np.maximum(raw["p10"], raw["p90"])
middle = np.clip(raw["p50"], lower, upper)
p10 = np.clip(np.expm1(lower), 0.0, None)
p50 = np.clip(np.expm1(middle), 0.0, None)
p90 = np.clip(np.expm1(upper), 0.0, None)

xgb_metrics = regression_metrics(actual, p50, naive, p10, p90)
print("XGBoost p50:", xgb_metrics)

Trained 3 quantile models in 15.4s
XGBoost p50: {'mae': 1.9205, 'wape': 0.5273, 'mase': 0.7771, 'coverage_90': 0.8375}


## 5. Holdout results

`WAPE` is the volume-weighted absolute error (the business-relevant metric: it
weights every unit, not every day). `MASE` compares against seasonal naive;
`coverage_90` is the share of actual demand inside `[p10, p90]` (target ≈ 0.90).


In [6]:
naive_metrics = regression_metrics(actual, naive)
croston_metrics = regression_metrics(actual, croston)
results = pd.DataFrame(
    [
        {
            "model": "XGBoost p50 (global quantile)",
            "mae": xgb_metrics["mae"],
            "wape": xgb_metrics["wape"],
            "mase": xgb_metrics["mase"],
            "coverage_90": xgb_metrics["coverage_90"],
        },
        {
            "model": "Seasonal naive (lag_7)",
            "mae": naive_metrics["mae"],
            "wape": naive_metrics["wape"],
            "mase": 1.0,
            "coverage_90": np.nan,
        },
        {
            "model": "Croston/SBA",
            "mae": croston_metrics["mae"],
            "wape": croston_metrics["wape"],
            "mase": round(croston_metrics["mae"] / naive_metrics["mae"], 4),
            "coverage_90": np.nan,
        },
    ]
)
display(results)  # noqa: F821 - provided by the Jupyter kernel

,model,mae,wape,mase,coverage_90
0,XGBoost p50 (global quantile),1.9205,0.5273,0.7771,0.8375
1,Seasonal naive (lag_7),2.4713,0.6785,1.0000,NaN
2,Croston/SBA,2.2163,0.6085,0.8968,NaN


## 6. Error analysis by ABC class

The aggregate metric hides the real modeling problem. Splitting the holdout by
ABC class shows where the error lives and why a single number is misleading.


In [7]:
products = pd.read_csv(REPO_ROOT / "data" / "raw" / "products.csv")
products = products[["product_id", "abc_class", "demand_pattern"]]
segment = valid.copy()
segment["actual"] = actual
segment["p50"] = p50
segment = segment.merge(products, on="product_id", how="left")

rows = []
for abc_class in ("A", "B", "C"):
    mask = segment["abc_class"].eq(abc_class).to_numpy()
    if not mask.any():
        continue
    rows.append(
        {
            "abc_class": abc_class,
            "series_days": int(mask.sum()),
            "zero_share": round(float((actual[mask] == 0).mean()), 3),
            "mae": round(float(np.abs(actual[mask] - p50[mask]).mean()), 4),
            "wape": round(
                float(np.abs(actual[mask] - p50[mask]).sum() / actual[mask].sum()), 4
            ),
        }
    )
display(pd.DataFrame(rows))  # noqa: F821

weekly = segment.copy()
weekly["week"] = weekly["date"].dt.to_period("W")
weekly = weekly.groupby(
    ["business_id", "branch_id", "product_id", "week"], as_index=False
)[["actual", "p50"]].sum()
weekly_wape = float(np.abs(weekly["actual"] - weekly["p50"]).sum() / weekly["actual"].sum())
print(f"Weekly WAPE (sum of 7 daily forecasts): {weekly_wape:.4f}")

,abc_class,series_days,zero_share,mae,wape
0,A,252,0.024,3.7857,0.2953
1,B,336,0.369,3.2876,0.6959
2,C,1092,0.651,1.0693,0.8978


Weekly WAPE (sum of 7 daily forecasts): 0.4115


## 7. Feature importance

Gain importance from the `p50` model gives a first, cheap explainability pass.
It is **not** causal and it is biased toward correlated lag features; SHAP is
used in `03_explainability.ipynb` for per-forecast attributions.


In [8]:
importance = pd.Series(
    models["p50"].feature_importances_, index=FEATURES, name="gain"
).sort_values(ascending=False)
group_share = pd.Series(
    {group: float(importance[features].sum()) for group, features in FEATURE_GROUPS.items()}
).sort_values(ascending=False)
print("Importance share by feature group:")
display(group_share.round(4).to_frame("share"))  # noqa: F821
print("Top individual features:")
display(importance.head(12).round(4).to_frame())  # noqa: F821

Importance share by feature group:


,share
Categorical encodings,0.9261
Demand history,0.0438
Inventory & price (lagged),0.0231
Calendar,0.0070


Top individual features:


,gain
demand_pattern_code,0.9108
roll_mean_28,0.0319
stockout_flag_lag1,0.0103
product_code,0.0101
lead_time_days,0.0049
branch_code,0.0033
dow,0.0032
roll_mean_14,0.0030
category_code,0.0019
roll_std_7,0.0019


## 8. Prophet baseline (optional)

Prophet is a strong baseline for **stable series with clear calendar
seasonality**, and the milestone asks for a real XGBoost/Prophet comparison. It
is fitted per series on the pre-holdout history with weekly and yearly
seasonality and evaluated on the same 28-day holdout.

The comparison is run on **all 60 series**, not a cherry-picked sample, so the
conclusion is honest: Prophet is competitive on the five highest-volume (smoother)
series, but its aggregate error is higher because many SKU-branch series are
zero-inflated and sparse. The dependency is optional (`pip install -e '.[ml]'`).


In [9]:
PROPHET_AVAILABLE = False
try:
    from prophet import Prophet

    PROPHET_AVAILABLE = True
except Exception as exc:  # pragma: no cover - optional dependency
    print(f"Prophet is not installed ({exc.__class__.__name__}); skipping the optional baseline.")
    print("Enable it with: .venv/bin/pip install -e '.[ml]'")

if PROPHET_AVAILABLE:
    import logging

    logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
    prophet_predictions = np.full(len(valid), np.nan)
    prophet_failures = 0
    started = time.time()
    for key, test in valid.groupby(list(SERIES_KEYS), sort=False):
        test = test.sort_values("date")
        positions = test.index.to_numpy()
        series = history[
            history["business_id"].eq(key[0])
            & history["branch_id"].eq(key[1])
            & history["product_id"].eq(key[2])
        ].sort_values("date")
        cutoff = first_dates.get(tuple(key))
        train_series = series.loc[series["date"] < cutoff, ["date", "units_sold"]].rename(
            columns={"date": "ds", "units_sold": "y"}
        )
        try:
            prophet_model = Prophet(
                weekly_seasonality=True,
                yearly_seasonality=True,
                daily_seasonality=False,
            )
            prophet_model.fit(train_series)
            future = pd.DataFrame({"ds": test["date"].to_numpy()})
            forecast = prophet_model.predict(future)
            prophet_predictions[positions] = np.clip(
                forecast["yhat"].to_numpy(dtype=float), 0.0, None
            )
        except Exception as exc:  # pragma: no cover - environment dependent
            prophet_failures += 1
            prophet_predictions[positions] = 0.0
            print(f"Prophet failed for {tuple(key)}: {exc.__class__.__name__}: {exc}")

    fitted_series = valid[list(SERIES_KEYS)].drop_duplicates().shape[0] - prophet_failures
    print(
        f"Prophet fitted {fitted_series} series in {time.time() - started:.1f}s "
        f"(failures: {prophet_failures})"
    )

    prophet_metrics = regression_metrics(
        actual, prophet_predictions, naive
    )
    comparison = pd.DataFrame(
        [
            {
                "model": "XGBoost p50 (global quantile)",
                "mae": xgb_metrics["mae"],
                "wape": xgb_metrics["wape"],
                "mase": xgb_metrics["mase"],
            },
            {
                "model": "Prophet (per series, all 60)",
                "mae": prophet_metrics["mae"],
                "wape": prophet_metrics["wape"],
                "mase": prophet_metrics["mase"],
            },
            {
                "model": "Croston/SBA",
                "mae": croston_metrics["mae"],
                "wape": croston_metrics["wape"],
                "mase": round(croston_metrics["mae"] / naive_metrics["mae"], 4),
            },
            {
                "model": "Seasonal naive (lag_7)",
                "mae": naive_metrics["mae"],
                "wape": naive_metrics["wape"],
                "mase": 1.0,
            },
        ]
    )
    display(comparison)  # noqa: F821

    top_keys = (
        valid.groupby(list(SERIES_KEYS))["units_sold"].sum().nlargest(5).reset_index()
    )
    sample_rows = []
    for _, key_row in top_keys.iterrows():
        mask = (
            valid["business_id"].eq(key_row["business_id"])
            & valid["branch_id"].eq(key_row["branch_id"])
            & valid["product_id"].eq(key_row["product_id"])
        ).to_numpy()
        sample_rows.append(
            {
                "product_id": key_row["product_id"],
                "branch_id": key_row["branch_id"],
                "prophet_mae": round(
                    float(np.abs(actual[mask] - prophet_predictions[mask]).mean()), 4
                ),
                "xgboost_mae": round(float(np.abs(actual[mask] - p50[mask]).mean()), 4),
            }
        )
    display(pd.DataFrame(sample_rows))  # noqa: F821
    print(
        "Prophet is competitive on the five highest-volume (smoother) series; "
        "on the full zero-inflated catalog the global XGBoost has the lower WAPE."
    )


Importing plotly failed. Interactive plots will not work.


14:20:04 - cmdstanpy - INFO - Chain [1] start processing


14:20:04 - cmdstanpy - INFO - Chain [1] done processing


14:20:04 - cmdstanpy - INFO - Chain [1] start processing


14:20:04 - cmdstanpy - INFO - Chain [1] done processing


14:20:04 - cmdstanpy - INFO - Chain [1] start processing


... (115 outputs left out)

Prophet fitted 60 series in 12.9s (failures: 0)


,model,mae,wape,mase
0,XGBoost p50 (global quantile),1.9205,0.5273,0.7771
1,"Prophet (per series, all 60)",2.0321,0.5579,0.8223
2,Croston/SBA,2.2163,0.6085,0.8968
3,Seasonal naive (lag_7),2.4713,0.6785,1.0000


,product_id,branch_id,prophet_mae,xgboost_mae
0,P-007,B-01,5.4075,6.4352
1,P-012,B-01,4.5390,4.3716
2,P-008,B-01,3.6153,4.2924
3,P-007,B-02,4.6567,4.6653
4,P-017,B-01,4.0990,3.8082


Prophet is competitive on the five highest-volume (smoother) series; on the full zero-inflated catalog the global XGBoost has the lower WAPE.


## 9. Decision, limitations and next steps

**Decision.** The global quantile XGBoost is the champion because it beats both
baselines on volume-weighted error, produces calibrated intervals for the
inventory policy and handles 60 sparse series with a single artifact. Prophet
remains the right tool for aggregate/stable series, not for this per-SKU panel.

**Known limitations.** The aggregate WAPE is dominated by C-class, zero-heavy
SKUs; the model is materially better on A-class items. The `p50` is a
conditional median: for lumpy SKUs it is often zero, which is correct for
absolute error but too low as a replenishment signal. Milestone M4 therefore
adds a planning signal that blends `p50` with `p90`, served through the
recommendation endpoint and evaluated with the inventory-policy simulation.

**Next milestones.** M4 moves this notebook into a reproducible training CLI
(`make train`), commits the model registry and serves it through FastAPI; M5 adds
drift monitoring; M6 adds the natural-language assistant.


In [10]:
print("Summary for the commit message:")
print(f"  XGBoost  MAE={xgb_metrics['mae']:.4f}  WAPE={xgb_metrics['wape']:.4f}  "
      f"MASE={xgb_metrics['mase']:.4f}  coverage_90={xgb_metrics['coverage_90']:.4f}")
print(f"  naive    WAPE={naive_metrics['wape']:.4f}")
print(f"  croston  WAPE={croston_metrics['wape']:.4f}")
print(f"  weekly WAPE={weekly_wape:.4f}")


Summary for the commit message:
  XGBoost  MAE=1.9205  WAPE=0.5273  MASE=0.7771  coverage_90=0.8375
  naive    WAPE=0.6785
  croston  WAPE=0.6085
  weekly WAPE=0.4115
